<a href="https://colab.research.google.com/github/nicholaspfeil/PokemonPredictorFork/blob/main/Ebay_Best_Card_Deal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests
import requests
import base64
import time
import pandas as pd

In [ ]:
from google.colab import userdata

EBAY_APP_ID = userdata.get("PokeData")
EBAY_CERT_ID= userdata.get("PokeDataCertID")

if not EBAY_APP_ID or not EBAY_CERT_ID:
    raise ValueError("Missing EBAY_APP_ID or EBAY_CERT_ID in Colab Secrets.")

def get_ebay_access_token():
    credentials = f"{EBAY_APP_ID}:{EBAY_CERT_ID}"

    encoded_credentials = base64.b64encode(
        credentials.encode("utf-8")
    ).decode("utf-8")

    headers = {
        "Authorization": f"Basic {encoded_credentials}",
        "Content-Type": "application/x-www-form-urlencoded"
    }

    data = {
        "grant_type": "client_credentials",
        "scope": "https://api.ebay.com/oauth/api_scope"
    }

    response = requests.post(
        "https://api.ebay.com/identity/v1/oauth2/token",
        headers=headers,
        data=data,
        timeout=20
    )

    response.raise_for_status()

    token_data = response.json()

    return token_data["access_token"]

token = get_ebay_access_token()
print("eBay authentication successful.")

eBay authentication successful.


In [ ]:
def search_ebay_cards(card_query, limit=20):

    token = get_ebay_access_token()

    url = "https://api.ebay.com/buy/browse/v1/item_summary/search"

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "X-EBAY-C-MARKETPLACE-ID": "EBAY_US"
    }

    params = {
        "q": card_query,
        "limit": limit,
        "sort": "price",
        "filter": "buyingOptions:{FIXED_PRICE}"
    }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=20
    )

    response.raise_for_status()

    return response.json()

In [ ]:
def ebay_results_to_dataframe(data):

    items = data.get("itemSummaries", [])

    rows = []

    for item in items:

        price_info = item.get("price", {})

        price = price_info.get("value")
        currency = price_info.get("currency")

        shipping = 0

        shipping_options = item.get("shippingOptions", [])

        if shipping_options:

            shipping_cost = shipping_options[0].get(
                "shippingCost",
                {}
            )

            shipping = float(
                shipping_cost.get("value", 0)
            )

        if price is not None:

            rows.append({
                "title": item.get("title"),
                "price": float(price),
                "shipping": shipping,
                "total_price": float(price) + shipping,
                "condition": item.get("condition"),
                "item_id": item.get("itemId"),
                "url": item.get("itemWebUrl"),
                "image": item.get("image", {}).get("imageUrl"),
                "currency": currency
            })

    df = pd.DataFrame(rows)

    if len(df) > 0:
        df = df.sort_values("total_price")

    return df


results = search_ebay_cards(
    "Charizard Obsidian Flames 228/197 PSA 9"
)

ebay_df = ebay_results_to_dataframe(results)

ebay_df.head()

,title,price,shipping,total_price,condition,item_id,url,image,currency
14,Charizard ex 228/197 SV03: Obsidian Flames Hyp...,59.99,0.0,59.99,Ungraded,v1|318774431567|0,https://www.ebay.com/itm/318774431567?_skw=Cha...,https://i.ebayimg.com/images/g/~1gAAeSw8NRqjL7...,USD
15,Charizard ex PSA 9 SV03 Obsidian Flames Hyper ...,60.00,0.0,60.00,Graded,v1|267740179029|0,https://www.ebay.com/itm/267740179029?_skw=Cha...,https://i.ebayimg.com/images/g/RgUAAeSwN09qZuO...,USD
16,Pokémon TCG Charizard EX Obsidian Flames 228/1...,65.00,0.0,65.00,Graded,v1|407125265692|0,https://www.ebay.com/itm/407125265692?_skw=Cha...,https://i.ebayimg.com/images/g/1TcAAeSwZoJqdMh...,USD
17,2023 English Pokemon Obsidian Flames HR Holo C...,67.77,0.0,67.77,Graded,v1|236880958709|0,https://www.ebay.com/itm/236880958709?_skw=Cha...,https://i.ebayimg.com/images/g/0tgAAeSwkvJqMZl...,USD
18,Pokémon Charizard ex SV03 Obsidian Flames Hype...,68.00,0.0,68.00,Graded,v1|147384784083|0,https://www.ebay.com/itm/147384784083?_skw=Cha...,https://i.ebayimg.com/images/g/ILQAAeSwPZZqNLX...,USD


In [ ]:
def find_best_ebay_deal(
    card_query,
    predicted_price,
    limit=20
):

    data = search_ebay_cards(
        card_query,
        limit=limit
    )

    df = ebay_results_to_dataframe(data)

    if df.empty:
        return None, df

    # Compare each listing to the model's estimated value
    df["discount_percent"] = (
        (predicted_price - df["total_price"])
        / predicted_price
        * 100
    )

    # Best deal = largest discount,
    # not simply the lowest absolute price
    df = df.sort_values(
        "discount_percent",
        ascending=False
    )

    best_listing = df.iloc[0]

    return best_listing, df

best, listings = find_best_ebay_deal(
card_query="Charizard Obsidian Flames 228/197 PSA 9",
predicted_price=120)

print(best)

title               Charizard ex 228/197 SV03: Obsidian Flames Hyp...
price                                                           59.99
shipping                                                          0.0
total_price                                                     59.99
condition                                                    Ungraded
item_id                                             v1|318774431567|0
url                 https://www.ebay.com/itm/318774431567?_skw=Cha...
image               https://i.ebayimg.com/images/g/~1gAAeSw8NRqjL7...
currency                                                          USD
discount_percent                                            50.008333
Name: 14, dtype: object


In [ ]:
def filter_reasonable_listings(df, predicted_price):

    if df.empty:
        return df

    # Remove listings with impossible/non-positive prices
    df = df[df["total_price"] > 0]

    # Don't automatically trust extreme outliers
    df = df[
        df["total_price"] >= predicted_price * 0.15
    ]

    return df

In [8]:
def filter_reasonable_listings(df, predicted_price):
    """Remove clearly unusable listings before ranking deals."""
    if df.empty:
        return df.copy()

    df = df.copy()

    # Remove invalid prices
    df = df[df["total_price"] > 0]

    # Avoid automatically treating extreme outliers as amazing deals.
    # This is intentionally conservative and can be tuned later.
    df = df[df["total_price"] >= predicted_price * 0.15]

    return df


def find_top_3_ebay_deals(
    card_query,
    predicted_price,
    limit=40
):
    """Return up to the top 3 eBay listings by discount vs. model value."""

    data = search_ebay_cards(
        card_query,
        limit=limit
    )

    df = ebay_results_to_dataframe(data)

    if df.empty:
        return df

    # Filter suspiciously low/out-of-range prices before ranking.
    df = filter_reasonable_listings(df, predicted_price)

    if df.empty:
        return df

    # Positive = listing is below the model's estimated value.
    df["discount_percent"] = (
        (predicted_price - df["total_price"])
        / predicted_price
        * 100
    )

    # A simple deal score. Higher is better.
    df["deal_score"] = df["discount_percent"].clip(lower=-100)

    # Best deal is the largest credible discount, not just the lowest price.
    df = df.sort_values(
        ["deal_score", "total_price"],
        ascending=[False, True]
    ).reset_index(drop=True)

    # Return only the best three options.
    return df.head(3)


# Example
TOP_QUERY = "Charizard Obsidian Flames 228/197 PSA 9"
PREDICTED_PRICE = 120

top_3_deals = find_top_3_ebay_deals(
    card_query=TOP_QUERY,
    predicted_price=PREDICTED_PRICE
)

top_3_deals[[
    "title",
    "total_price",
    "discount_percent",
    "condition",
    "url"
]]

def print_top_3_deals(top_3_df, predicted_price):
    """Display the top three deals in a readable format."""

    if top_3_df.empty:
        print("No suitable eBay listings were found.")
        return

    print(f"Estimated model value: ${predicted_price:,.2f}\n")

    for rank, (_, row) in enumerate(top_3_df.iterrows(), start=1):
        discount = row["discount_percent"]

        if discount > 0:
            verdict = f"{discount:.1f}% below model value"
        elif discount < 0:
            verdict = f"{abs(discount):.1f}% above model value"
        else:
            verdict = "at model value"

        print(f"#{rank}")
        print(f"  {row['title']}")
        print(f"  Total price: ${row['total_price']:,.2f} ({verdict})")
        print(f"  Condition: {row['condition']}")
        print(f"  Link: {row['url']}")
        print()


print_top_3_deals(top_3_deals, PREDICTED_PRICE)

Estimated model value: $120.00

#1
  Charizard ex 228/197 SV03: Obsidian Flames Hyper Rare MINT PSA 9
  Total price: $59.99 (50.0% below model value)
  Condition: Ungraded
  Link: https://www.ebay.com/itm/318774431567?_skw=Charizard+Obsidian+Flames+228%2F197+PSA+9&hash=item4a386fcf4f:g:~1gAAeSw8NRqjL7L

#2
  Charizard ex PSA 9 SV03 Obsidian Flames Hyper Rare Tera Holo 228/197
  Total price: $60.00 (50.0% below model value)
  Condition: Graded
  Link: https://www.ebay.com/itm/267740179029?_skw=Charizard+Obsidian+Flames+228%2F197+PSA+9&hash=item3e568eea55:g:RgUAAeSwN09qZuOR

#3
  Pokémon TCG Charizard EX Obsidian Flames 228/197 psa 9
  Total price: $65.00 (45.8% below model value)
  Condition: Graded
  Link: https://www.ebay.com/itm/407125265692?_skw=Charizard+Obsidian+Flames+228%2F197+PSA+9&hash=item5eca8e8d1c:g:1TcAAeSwZoJqdMhw

